In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

In [2]:
# 1. Carregar o modelo pré-treinado
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

In [3]:
from datasets import Dataset

# Carregar texto bruto
with open("txts/arquivo_convertido.txt", "r", encoding="utf-8") as f:
    data = f.read()

# Normalizar divisões entre parágrafos
normalized_data = data.replace("\n", "\n\n").replace("\n\n\n", "\n\n")  # Garante divisões consistentes

# Dividir em parágrafos
paragraphs = [p.strip() for p in normalized_data.split("\n\n") if p.strip()]  # Remove espaços e linhas vazias

# Criar dataset
dataset = Dataset.from_dict({"text": paragraphs})

# Verificar o dataset
print(dataset)
print(len(dataset))

Dataset({
    features: ['text'],
    num_rows: 66613
})
66613


In [4]:
print(dataset["text"][:5])

['UNIVERSIDADE FEDERAL DE SERGIPE', 'reitor', 'Prof. Dr. Angelo Roberto Antoniolli', 'vice-reitor', 'Prof. Dr. André Maurício Conceição de Souza']


In [6]:
split_datasets = dataset.train_test_split(test_size=0.3)  # 10% para validação
print(split_datasets)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 46629
    })
    test: Dataset({
        features: ['text'],
        num_rows: 19984
    })
})


In [15]:
# Função de tokenização
def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Aplicar a tokenização
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Dividir o dataset tokenizado em treino e validação
split_datasets = tokenized_datasets.train_test_split(test_size=0.3)

# Verificar o dataset tokenizado
print(split_datasets["train"].column_names)

Map:   0%|          | 0/66613 [00:00<?, ? examples/s]

['text', 'input_ids', 'attention_mask', 'labels']


In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    eval_steps=50,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=5e-6,
    num_train_epochs=6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    remove_unused_columns=True,  # Ignorar colunas desnecessárias
)

/home/jimi/unicamp/phd_projecs/proj_doc/venv/lib/python3.11/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
# Configurar o Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
)

In [ ]:
# Treinar o modelo
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
